# Company valuation template

Copy this notebook into a **local working folder** and rename it to the ticker (for example `HROW.ipynb`). Fill every `REPLACE` / `TODO`, uncomment only the methods that apply, and run the cells from that folder.

`import finagle` works from any folder after a one-time install from the repo (`pip install -e .` or `conda develop .`). You do not run this notebook from the repo root.

`display_fin()` writes `{ticker}.xlsx` in the working folder. The class also writes `{ticker}.log` there. The Excel layout comes from `company_template.xlsx` next to the installed package, not from this folder.

The instance below is named `co`. The `ticker` string is what names the Excel file and the log.

## Where each method goes

Run the steps in this order. Later steps assume the earlier ones have already been run.

1. **`company(...)`** — cost of capital, tax, shares, terminal growth, horizon, dividend policy. Optional: pass `fcfe`, `fcff`, or `fcf` for a direct DCF and skip the forecast, load, and `fcf_from_*` steps. That path still needs a `financials` dict with at least `date`.
2. **Before load**, mutate the dict: `forecast_ebitda`, then `forecast_capex`, then `forecast_sbc`. Comment `forecast_ebitda` when `ebitda` is already a full list of `year + 1` values. Keep `revenue` at `[0]` in that case. `forecast_capex` and `forecast_sbc` leave a list alone when it is already that long.
3. **`load_financials`** — if you did not pass `financials` into the constructor.
4. **Build free cash flow** — `fcf_from_ebitda` (usual) **or** `fcf_from_earnings` (needs an `e` key). Do not use both.
5. **Capital actions**, only after FCF exists, in this order:
   - `fcf_to_acquire` — once per deal; recalculates FCF
   - `noa_to_dispose` — sell non-operating assets; recalculates FCF
   - `fcf_to_debt` — path debt toward a Debt/EBITDA target; recalculates FCF
   - then **exactly one** distribution method:
     - `fcf_to_allocate` — buybacks and/or balance-sheet cash, then dividends (calls `fcf_to_bs`)
     - `fcf_to_buyback` — use FCFE for buybacks only
     - `fcf_to_bs` — leave undistributed FCFE on `cashBS` and pay it out as a terminal dividend
6. **`value()`** — after allocation is final. Discounts FCFE, FCFF, and dividends.
7. **`display_fin()`** — last. Styled table plus `{ticker}.xlsx`.

## Step by step

1. Run the import cell.
2. Fill the notes: ticker, reporting date, and the thesis for this case.
3. Set the constructor inputs in the model cell.
   - `rd` cost of debt, `re` cost of equity, `t` marginal tax rate.
   - `te` year-1 effective tax rate. Leave `None` to derive it inside `fcf_from_ebitda`.
   - `shares` shares outstanding (all classes). `price` current share price, used by buybacks.
   - `gt` terminal growth. `roict` terminal ROIC, used mainly for terminal depreciation.
   - `year` number of forecast years after the baseline year. `dividend` current dividend policy (a number or a list).
4. Fill `financials`. Index 0 is the trailing-twelve-months / baseline year. `date` is `'%Y-%m-%d'`.
   - `ebitda`: the TTM value only, **or** a full list of `year + 1` values. A full list is the forecast. Comment out `forecast_ebitda` or that call replaces the list.
   - `revenue`: leave `[0]`. `forecast_ebitda` overwrites it when that call is active. If you comment `forecast_ebitda`, keep `[0]`. `value()` needs the column either way.
   - `dwc` and `debt` need `year + 1` entries.
   - `capex` and `sbc` can be one value, a partial list, or a full list. `forecast_capex` and `forecast_sbc` extend a short list. A list that is already `year + 1` long is left as you typed it.
   - One value is enough for `tax`, `da`, `interest`, `cash`, `nol`, and `noa`. Leave `da` as TTM only when later years should be calculated from capex and `roict`.
5. EBITDA. Use at most one `forecast_ebitda` line (growth-only or margin). Comment both lines when `ebitda` is already a full list. Leave `forecast_capex` and `forecast_sbc` active unless you want to skip that item; a complete list is not extended.
6. `load_financials(financials.copy())`, then `fcf_from_ebitda()`.
7. Uncomment capital-action lines in the order in the method map. Use only one of `fcf_to_allocate`, `fcf_to_buyback`, or `fcf_to_bs`.
8. `value()`, then `display_fin()`.
9. Check the table: cash should not go negative (the class does not model a capital raise; fund a cash need with debt). Treat negative FCFE after year 1 with caution. Year-1 negative FCFE can be offset by year-0 cash. Once you allocate cash to buybacks or the balance sheet, per-share DDM and discounted FCFE will differ.

### Share price and buybacks

`dp` is used by `fcf_to_buyback` and `fcf_to_allocate`.

- `'proportional'` (default) holds the year-1 forward EV/EBITDA multiple and updates the price each year.
- `'constant'` keeps the share price fixed at the `price` you pass.

`buybacks` is used by `fcf_to_allocate`.

- `None` buys back as much as FCFE allows.
- A list or a number is a fixed buyback schedule. Cash left over is stored on the balance sheet.
- `0` skips buybacks and leaves cash for dividends / `cashBS`.

In [ ]:
import finagle as cmp

## Notes

- **Ticker:** REPLACE
- **Reporting date:** REPLACE
- **Thesis:**
  - REPLACE

In [ ]:
# --- 3. Constructor ---
ticker = 'REPLACE'   # TODO: symbol; names REPLACE.xlsx and REPLACE.log
rd = 0.05            # TODO: cost of debt
re = 0.09            # TODO: cost of equity
t = 0.21             # TODO: marginal tax rate
te = None            # year-1 effective tax rate; None derives it in fcf_from_ebitda
shares = 1           # TODO: shares outstanding, all classes
price = 0            # TODO: current share price (buybacks / allocate)
gt = 0.02            # TODO: terminal growth
roict = 0.15         # TODO: terminal ROIC (terminal depreciation)
year = 10            # forecast years after the baseline year
dividend = [0]       # TODO: current dividend policy (number or list)

co = cmp.company(
    ticker=ticker, rd=rd, re=re, t=t, te=te,
    shares=shares, price=price, gt=gt, roict=roict,
    year=year, dividend=dividend,
)
# Direct DCF (step 1 alternative): also pass fcfe=, fcff=, or fcf=, and a
# financials dict with at least 'date'. Then skip forecast_*, load_financials,
# and fcf_from_* and go to value().

# --- 4. Financials ---
# Index 0 is the baseline / TTM year.
# debt and dwc must have year + 1 entries.
# ebitda is either TTM only, or a full list of year + 1 values.
# If ebitda is a full list, comment out forecast_ebitda below. That call replaces the list.
# Keep revenue at [0] when forecast_ebitda is commented. value() needs the column.
# forecast_ebitda overwrites revenue when that call is left active.
# capex and sbc may be partial. A list that is already year + 1 long is left unchanged.
n = year + 1
financials = {
    'date': 'REPLACE',   # TODO: last reporting date, '%Y-%m-%d'
    'revenue': [0],      # leave [0] if forecast_ebitda is commented; that call overwrites it
    'ebitda': [0],       # TODO: TTM only, or a full list of year + 1 values
    'capex': [0],        # TODO: TTM, partial, or full capex
    'dwc': [0] * n,      # TODO: change in non-cash working capital, full horizon
    'sbc': [0],          # TODO: TTM, partial, or full stock-based compensation
    'tax': [0],          # TODO: TTM reported taxes
    'da': [0],           # TODO: TTM D&A only; later years come from capex and roict
    'debt': [0] * n,     # TODO: short + long-term debt, full horizon
    'interest': [0],     # TODO: TTM interest; later years = rd * prior debt
    'cash': 0,           # TODO: excess cash / short-term investments (year 0)
    'nol': 0,            # TODO: net operating loss carryforwards
    'noa': 0,            # TODO: non-operating assets
}

# --- 5. Forecast EBITDA and revenue ---
# forecast_ebitda(ebitda_ttm, gf, financials=None, me=None, mc=None, gsnext=None)
# writes financials['ebitda'] and financials['revenue'].
#
# ebitda_ttm: last-year / TTM EBITDA. This becomes index 0 of the series.
# gf: explicit EBITDA growth rate(s), a float or a list (0.10 means +10%).
#     Rates are interpolated from the last explicit rate toward terminal growth gt.
#     Only the non-NaN prefix of a list counts as explicit years.
#
# Growth-only (leave me, mc, and gsnext unset, or omit any one of them):
#     ebitda[i+1] = ebitda[i] * (1 + g[i])
#     revenue is filled with 0.
#
# Margin path (only when me, mc, and gsnext are all set):
#     me: EBITDA margin (EBITDA / sales) at the end of the explicit gf years.
#         Next year's revenue is ebitda / me * (1 + gs).
#     mc: contribution margin (incremental EBITDA / incremental sales) at that handoff.
#     gsnext: sales growth in the first year after the explicit EBITDA growth rates.
#             Sales growth is then interpolated from gsnext toward gt.
#     During the explicit gf years, EBITDA still compounds by g and revenue is left unset.
#     After that:
#         ebitda[i+1] = ebitda[i] * (1 + (mc / me) * gs[i])
#         revenue is the sales level implied by me and gs
#         me is updated to the new EBITDA / revenue, so the margin fades with sales growth.

ebitda_ttm = financials['ebitda'][0]
gf = [0.10, 0.05]  # TODO: explicit EBITDA growth rates

# Growth-only path (active).
# Comment this out when ebitda above is already a full list of year + 1 values.
# Also comment it out if you use the margin path instead.
# Commenting this call does not remove revenue; leave revenue at [0].
co.forecast_ebitda(ebitda_ttm, gf, financials)

# Margin path. Uncomment this and comment the growth-only call above.
# me = 0.25       # EBITDA / sales at the end of the explicit gf years
# mc = 0.40       # incremental EBITDA / incremental sales at that handoff
# gsnext = 0.10   # sales growth in the first year after explicit EBITDA growth
# co.forecast_ebitda(ebitda_ttm, gf, financials, me=me, mc=mc, gsnext=gsnext)

# Holds the last capex/EBITDA ratio for years beyond the list you pass.
co.forecast_capex(financials['capex'], financials)

# Interpolates SBC/EBITDA toward sbc_rate_t.
# sbc_rate_t=None holds the last explicit year's SBC/EBITDA ratio.
co.forecast_sbc(sbc_f=financials['sbc'], sbc_rate_t=None, financials=financials)  # TODO: set sbc_rate_t

# --- 6. Load and build FCF ---
co.load_financials(financials=financials.copy())
co.fcf_from_ebitda()

# Earnings path (step 4 alternative). Do not combine with the EBITDA block above.
# Requires financials['e']. Terminal payout = 1 - gt/ROE.
# co.fcf_from_earnings(payout=1, gf=0, ROE=1)

# --- 7. Capital actions (uncomment only what applies) ---
# Order: acquisitions, disposals, debt, then exactly one distribution method.
# fcf_to_acquire, noa_to_dispose, and fcf_to_debt each recalculate FCF.

# co.fcf_to_acquire(adjust_cash=False, year_a=1, ebitda_frac=0.10, multiple=10, leverage=3, gnext=0.10, cap_frac=0.20)
# co.fcf_to_acquire(adjust_cash=False, year_a=2, ebitda_frac=0.10, multiple=10, leverage=3, gnext=0.10, cap_frac=0.20)

# co.noa_to_dispose(dnoa=0, tax=0, year_dis=1)  # TODO: gross disposal; tax is a fraction

# co.fcf_to_debt(leverage=3, year_d=1)  # target Debt/EBITDA, starting at year_d

# Distribution: leave only one of the next three uncommented.
# dp='proportional' holds the year-1 forward EV/EBITDA multiple.
# dp='constant' keeps the share price fixed at price.
# buybacks=None buys back as much as FCFE allows.
# buybacks as a list or number is a fixed schedule; leftover cash goes to the balance sheet.
# buybacks=0 skips buybacks and leaves cash for dividends / cashBS.
# co.fcf_to_allocate(price=price, dp='proportional', buybacks=None)
# co.fcf_to_buyback(price=price, dp='proportional')
# co.fcf_to_bs()

# --- 8. Value and report ---
co.value()
co.display_fin()

## Scenario copy

Duplicate the cell below when you want another case (different growth, a margin path, a deal, or a leverage target). The first variant to try is the margin path: uncomment that `forecast_ebitda` call and comment out the growth-only call.

In [ ]:
# Scenario copy. Edit this cell; leave the cell above as the base case.
# First variant: uncomment the margin-path forecast_ebitda and comment the growth-only call.

ticker = 'REPLACE'
rd = 0.05
re = 0.09
t = 0.21
te = None
shares = 1
price = 0
gt = 0.02
roict = 0.15
year = 10
dividend = [0]

co = cmp.company(
    ticker=ticker, rd=rd, re=re, t=t, te=te,
    shares=shares, price=price, gt=gt, roict=roict,
    year=year, dividend=dividend,
)

n = year + 1
financials = {
    'date': 'REPLACE',
    'revenue': [0],  # leave [0] if forecast_ebitda is commented; that call overwrites it
    'ebitda': [0],   # TTM only, or a full list of year + 1 values
    'capex': [0],
    'dwc': [0] * n,
    'sbc': [0],
    'tax': [0],
    'da': [0],
    'debt': [0] * n,
    'interest': [0],
    'cash': 0,
    'nol': 0,
    'noa': 0,
}

# gf: explicit EBITDA growth. Interpolated toward gt. Non-NaN prefix = explicit years.
# Growth-only: ebitda[i+1] = ebitda[i] * (1 + g[i]); revenue is 0.
# Margin path (me, mc, and gsnext all set), after the explicit gf years:
#   me = EBITDA/sales at the handoff; revenue = ebitda/me * (1+gs)
#   mc = incremental EBITDA / incremental sales
#   gsnext = sales growth in the first year after explicit EBITDA growth, then faded to gt
#   ebitda[i+1] = ebitda[i] * (1 + (mc/me) * gs[i]); me is updated to the new margin
ebitda_ttm = financials['ebitda'][0]
gf = [0.10, 0.05]

# Comment this out when ebitda is already a full list. Keep revenue at [0] in that case.
co.forecast_ebitda(ebitda_ttm, gf, financials)

# me = 0.25
# mc = 0.40
# gsnext = 0.10
# co.forecast_ebitda(ebitda_ttm, gf, financials, me=me, mc=mc, gsnext=gsnext)

co.forecast_capex(financials['capex'], financials)
co.forecast_sbc(sbc_f=financials['sbc'], sbc_rate_t=None, financials=financials)

co.load_financials(financials=financials.copy())
co.fcf_from_ebitda()
# co.fcf_from_earnings(payout=1, gf=0, ROE=1)  # alternate path; do not combine with EBITDA

# co.fcf_to_acquire(adjust_cash=False, year_a=1, ebitda_frac=0.10, multiple=10, leverage=3, gnext=0.10, cap_frac=0.20)
# co.fcf_to_acquire(adjust_cash=False, year_a=2, ebitda_frac=0.10, multiple=10, leverage=3, gnext=0.10, cap_frac=0.20)
# co.noa_to_dispose(dnoa=0, tax=0, year_dis=1)
# co.fcf_to_debt(leverage=3, year_d=1)

# Exactly one distribution method:
# co.fcf_to_allocate(price=price, dp='proportional', buybacks=None)
# co.fcf_to_buyback(price=price, dp='proportional')
# co.fcf_to_bs()

co.value()
co.display_fin()